# EasyOCR

In [ ]:
import easyocr
import cv2
import time
import csv
from ultralytics import YOLO

# --- MODELOS ---
model_coche = YOLO('yolo11n.pt')
model_plate = YOLO('./runs/detect/train/weights/best.pt')

# --- OCR ---
reader = easyocr.Reader(['es'], gpu=True)

# --- VIDEO ---
capture_video = cv2.VideoCapture("./videos/video5.mp4")

# Variables para estadísticas
tiempos_inferencia_yolo = []
tiempos_inferencia_easyocr = []

# Configuración de salida
frame_width = int(capture_video.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(capture_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = capture_video.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_video = cv2.VideoWriter('./videos/output_video_easyocr.mp4', fourcc, fps, (frame_width, frame_height))

# --- CSV ---
csv_filename = "./CSVs/deteccion_de_matricula_easyocr.csv"
csv_header = [
    "fotograma", "tipo_objeto", "confianza_deteccion",
    "x1", "y1", "x2", "y2",
    "matrícula_detectada", "x1_matrícula", "y1_matrícula", "x2_matrícula", "y2_matrícula",
    "texto_matricula_ocr", "tiempo_inferencia_yolo", "tiempo_inferencia_easyocr"
]

with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    csv_writer = csv.writer(csvfile, delimiter=';')
    csv_writer.writerow(csv_header)

    frame_count = 0
    while True:
        ret, frame_video = capture_video.read()
        if not ret:
            break
        frame_count += 1

        start_time_yolo = time.perf_counter()
        # Detectamos solo clases de vehículos
        vehicle_results = model_coche(frame_video, stream=True, classes=[2, 3, 5, 7], conf=0.7)  # car, motorbike, bus, truck
        end_time_yolo = time.perf_counter()
        tiempo_inferencia_yolo = end_time_yolo - start_time_yolo
        tiempos_inferencia_yolo.append(tiempo_inferencia_yolo)

        frame_detections = []

        # Recorremos detecciones
        for result in vehicle_results:
            for v in result.boxes:
                class_id = int(v.cls[0])
                class_name = model_coche.names[class_id]
                confidence = float(v.conf[0])

                x1, y1, x2, y2 = map(int, v.xyxy[0])
                vehicle_roi = frame_video[y1:y2, x1:x2]

                plate_text = ""
                lp_x1 = lp_y1 = lp_x2 = lp_y2 = ""
                tiempo_inferencia_easyocr = 0.0

                # --- DETECCIÓN DE MATRÍCULA ---
                if vehicle_roi.size > 0:
                    plate_results = model_plate(vehicle_roi, stream=True)
                    for p in plate_results:
                        for box in p.boxes:
                            px1, py1, px2, py2 = map(int, box.xyxy[0])
                            lp_x1, lp_y1, lp_x2, lp_y2 = x1 + px1, y1 + py1, x1 + px2, y1 + py2
                            plate_roi = frame_video[lp_y1:lp_y2, lp_x1:lp_x2]

                            if plate_roi.size > 0:
                                start_time_easyocr = time.perf_counter()
                                ocr_result = reader.readtext(
                                    plate_roi,
                                    allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
                                    detail=0
                                )
                                end_time_easyocr = time.perf_counter()
                                tiempo_inferencia_easyocr = end_time_easyocr - start_time_easyocr
                                tiempos_inferencia_easyocr.append(tiempo_inferencia_easyocr)

                                if ocr_result:
                                    plate_text = "".join(ocr_result).replace(" ", "")
                                    print(f"Matrícula detectada: {plate_text}")

                                    cv2.rectangle(frame_video, (lp_x1, lp_y1), (lp_x2, lp_y2), (0, 0, 255), 2)
                                    cv2.putText(frame_video, plate_text, (lp_x1, lp_y1 - 10),
                                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

                # Dibujar el vehículo
                cv2.rectangle(frame_video, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame_video, f"{class_name} {confidence:.2f}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

                # Información de tiempos
                time_info = f"YOLO: {tiempo_inferencia_yolo:.3f}s"
                if tiempo_inferencia_easyocr > 0:
                    time_info += f" | EasyOCR: {tiempo_inferencia_easyocr:.3f}s"
                cv2.putText(frame_video, time_info, (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(frame_video, f"Frame: {frame_count}", (10, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

                # Guardar datos del frame
                row_data = [
                    frame_count, class_name, confidence,
                    x1, y1, x2, y2,
                    "SÍ" if plate_text else "NO",
                    lp_x1, lp_y1, lp_x2, lp_y2,
                    plate_text,
                    f"{tiempo_inferencia_yolo:.6f}",
                    f"{tiempo_inferencia_easyocr:.6f}"
                ]
                frame_detections.append(row_data)

        # Guardar fuera del bucle interno
        if frame_detections:
            csv_writer.writerows(frame_detections)
            out_video.write(frame_video)

capture_video.release()
out_video.release()

print("\n✅ Procesamiento completado.")
print(f"📹 Video guardado como: output_video_easyocr.mp4")
print(f"📄 CSV guardado como: {csv_filename}")


c:\Users\andsa\anaconda3\envs\easyOCR\lib\site-packages\easyocr\utils.py:9: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 1.21.5)
  from scipy import ndimage



0: 384x640 (no detections), 217.3ms
Speed: 3.7ms preprocess, 217.3ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 15.0ms
Speed: 1.3ms preprocess, 15.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 13.3ms
Speed: 0.9ms preprocess, 13.3ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 13.3ms
Speed: 1.1ms preprocess, 13.3ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 14.4ms
Speed: 1.0ms preprocess, 14.4ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 13.5ms
Speed: 1.1ms preprocess, 13.5ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 14.5ms
Speed: 1.0ms preprocess, 14.5ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 14.0ms
Speed: 1.3ms preprocess, 14.0ms

# Tesseract

In [1]:
import pytesseract
import cv2
import time
import csv
from ultralytics import YOLO
import numpy as np
import re

# --- CONFIGURACIÓN DE TESSERACT ---
pytesseract.pytesseract.tesseract_cmd = r'C:/Program Files/Tesseract-OCR/tesseract.exe'

# --- MODELOS ---
model_coche = YOLO('yolo11n.pt')
model_plate = YOLO('./runs/detect/train/weights/best.pt')

# --- VIDEO ---
capture_video = cv2.VideoCapture("./videos/video5.mp4")

# --- ESTADÍSTICAS ---
tiempos_inferencia_yolo = []
tiempos_inferencia_tesseract = []

# --- CONFIGURACIÓN VIDEO DE SALIDA ---
frame_width = int(capture_video.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(capture_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = capture_video.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_video = cv2.VideoWriter('./videos/output_video_tesseract.mp4', fourcc, fps, (frame_width, frame_height))

# --- CSV ---
csv_filename = "./CSVs/deteccion_de_matricula_tesseract.csv"
csv_header = [
    "fotograma", "tipo_objeto", "confianza_deteccion",
    "x1", "y1", "x2", "y2",
    "matrícula_detectada", "x1_matrícula", "y1_matrícula", "x2_matrícula", "y2_matrícula",
    "texto_matricula_ocr", "tiempo_inferencia_yolo", "tiempo_inferencia_tesseract"
]

with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    csv_writer = csv.writer(csvfile, delimiter=';')
    csv_writer.writerow(csv_header)

    frame_count = 0
    while True:
        ret, frame_video = capture_video.read()
        if not ret:
            break
        frame_count += 1

        # --- DETECCIÓN VEHÍCULOS ---
        start_time_yolo = time.perf_counter()
        vehicle_results = model_coche(frame_video, stream=True, classes=[2, 3, 5, 7], conf=0.7)
        end_time_yolo = time.perf_counter()
        tiempo_inferencia_yolo = end_time_yolo - start_time_yolo
        tiempos_inferencia_yolo.append(tiempo_inferencia_yolo)

        frame_detections = []

        # --- PROCESAR CADA VEHÍCULO DETECTADO ---
        for result in vehicle_results:
            for v in result.boxes:
                class_id = int(v.cls[0])
                class_name = model_coche.names[class_id]
                confidence = float(v.conf[0])

                x1, y1, x2, y2 = map(int, v.xyxy[0])
                vehicle_roi = frame_video[y1:y2, x1:x2]

                plate_text = ""
                lp_x1_frame = lp_y1_frame = lp_x2_frame = lp_y2_frame = 0
                tiempo_inferencia_tesseract = 0.0

                # --- DETECCIÓN MATRÍCULA DENTRO DEL VEHÍCULO ---
                if vehicle_roi.size > 0:
                    plate_results = model_plate(vehicle_roi, stream=True)
                    for p in plate_results:
                        for box in p.boxes:
                            px1, py1, px2, py2 = map(int, box.xyxy[0])
                            
                            # ROI de la matrícula relativo al vehículo
                            plate_roi = vehicle_roi[py1:py2, px1:px2]

                            # Coordenadas absolutas para dibujar sobre el frame completo
                            lp_x1_frame, lp_y1_frame = x1 + px1, y1 + py1
                            lp_x2_frame, lp_y2_frame = x1 + px2, y1 + py2

                            if plate_roi.size > 0:
                                start_time_tesseract = time.perf_counter()
                                try:
                                    # OCR usando Tesseract
                                    plate_text_raw = pytesseract.image_to_string(
                                        plate_roi,
                                        config='--psm 8 --oem 3 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
                                    )
                                    # Limpiar el texto
                                    plate_text = re.sub(r'[^A-Z0-9]', '', plate_text_raw).strip()
                                except Exception:
                                    plate_text = "OCR_Error"
                                end_time_tesseract = time.perf_counter()
                                tiempo_inferencia_tesseract = end_time_tesseract - start_time_tesseract
                                tiempos_inferencia_tesseract.append(tiempo_inferencia_tesseract)

                                # Dibujar matrícula detectada
                                if plate_text:
                                    print(f"Matrícula detectada: {plate_text}")
                                    cv2.rectangle(frame_video, (lp_x1_frame, lp_y1_frame), (lp_x2_frame, lp_y2_frame), (0, 0, 255), 2)
                                    cv2.putText(frame_video, plate_text, (lp_x1_frame, lp_y1_frame - 10),
                                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

                # --- DIBUJAR VEHÍCULO ---
                cv2.rectangle(frame_video, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame_video, f"{class_name} {confidence:.2f}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

                # --- INFORMACIÓN DE TIEMPOS ---
                time_info = f"YOLO: {tiempo_inferencia_yolo:.3f}s"
                if tiempo_inferencia_tesseract > 0:
                    time_info += f" | OCR: {tiempo_inferencia_tesseract:.3f}s"
                cv2.putText(frame_video, time_info, (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(frame_video, f"Frame: {frame_count}", (10, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

                # --- GUARDAR EN CSV ---
                row_data = [
                    frame_count, class_name, confidence,
                    x1, y1, x2, y2,
                    "SÍ" if plate_text else "NO",
                    lp_x1_frame, lp_y1_frame, lp_x2_frame, lp_y2_frame,
                    plate_text,
                    f"{tiempo_inferencia_yolo:.6f}",
                    f"{tiempo_inferencia_tesseract:.6f}"
                ]
                frame_detections.append(row_data)

        # --- GUARDAR RESULTADOS DEL FRAME ---
        if frame_detections:
            csv_writer.writerows(frame_detections)
            out_video.write(frame_video)

capture_video.release()
out_video.release()
print("\n✅ Procesamiento completado con Tesseract OCR.")
print(f"📹 Video guardado como: output_video_tesseract.mp4")
print(f"📄 CSV guardado como: {csv_filename}")



0: 384x640 (no detections), 174.1ms
Speed: 43.6ms preprocess, 174.1ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 26.1ms
Speed: 1.4ms preprocess, 26.1ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 13.3ms
Speed: 1.3ms preprocess, 13.3ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 14.0ms
Speed: 1.3ms preprocess, 14.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 21.2ms
Speed: 1.4ms preprocess, 21.2ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 25.8ms
Speed: 1.9ms preprocess, 25.8ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 21.2ms
Speed: 1.4ms preprocess, 21.2ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 27.3ms
Speed: 1.1ms preprocess, 27.3m